<a href="https://colab.research.google.com/github/yashb98/90Days_Machine_learinng/blob/main/Intro_to_LLMs_Building_a_RAG_system_(Project_9).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Retrieval-Augmented Generation (RAG) is a method that combines retrieval and generation:

**(1) Retrieval**

•	This is fetching relevant information from your own documents.

•	Done using vector embeddings + vector database.

•	Helps the model focus on relevant information instead of generating purely from training data.

**(2) Generation**


•	Uses a large language model (LLM) like GPT to generate answers based on retrieved content.

•	Ensures the answers are grounded in actual data, reducing hallucinations.

**Why it matters:**

•	A regular LLM can make up information.

•	RAG ensures answers are backed by your own documents, making it suitable for research papers, manuals, or any custom knowledge base.

## Install Dependencies


### This cell ensures all necessary tools are available for building the pipeline by installing or updating required Python packages:

1. **langchain:** A framework for simplifying the creation of applications that use LLMs by chaining together various components.

2. **langchain-community:** Provides integrations for external third-party resources, such as document loaders, vector stores, and specific model bindings.

3. **langchain-text-splitters:** A utility library dedicated to breaking down large documents into smaller, manageable chunks. This addresses the LLM's context length limitation.

4. **sentence-transformers:** A specialized library that provides models to convert text into fixed-size numerical vectors, known as embeddings. These embeddings are vital for the semantic search component of the Retrieval step.

5. **faiss-cpu:** A library by Facebook AI for performing efficient similarity search across large datasets of dense vectors, serving as the Vector Database in a RAG pipeline.

6. **pypdf:** A pure-Python tool used to read and extract textual data from PDF files.

In [1]:
!pip install -U langchain langchain-community langchain-text-splitters sentence-transformers faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.3/469.3 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.6/207.6 kB 22.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing 

## Import Modules

### This imports the necessary classes and functions from the installed libraries:

PyPDFLoader: Instantiated to load the contents of the PDF document.

RecursiveCharacterTextSplitter: The chosen implementation for breaking the input document into chunks.

SentenceTransformerEmbeddings: Used to select and load a sentence transformer model for generating semantic vectors (embeddings).

FAISS: Imported to create the in-memory index for fast similarity search across the vector embeddings.

os: A standard Python module for interfacing with the operating system, often used for file path operations.

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import FAISS
import os

## Load your PDF document

Here I am using a book for builduing AI Engineering applications ( by Chip Huyen )



#### This is the first data processing step, taking the raw file and turning it into data objects:

1. pdf_path = ...: Defines the location of the source document ("AI Engineering" by Chip Huyen).

2. loader = PyPDFLoader(pdf_path): Creates a loader object tied to the document path.

3. pages = loader.load(): Executes the PDF extraction, resulting in a list where each element is a LangChain Document object corresponding to a single page from the PDF.

**Purpose:** The output confirms 991 pages were loaded, verifying successful data access and text extraction.

In [3]:
# Load PDF pages
pdf_path = "/content/_OceanofPDF.com_AI_Engineering_Building_Applications_-_Chip_Huyen.pdf"
loader = PyPDFLoader(pdf_path)
pages = loader.load()

print(f"Total pages loaded: {len(pages)}")
print("Sample page content:\n")
print(pages[3].page_content[:500])

Total pages loaded: 991
Sample page content:

AI Engineering is a comprehensive guide that serves as an essential
reference for both understanding and implementing AI systems in
practice.
—Han Lee, Director—Data Science, Moody’s
AI Engineering is an essential guide for anyone building software
with Generative AI! It demystifies the technology, highlights the
importance of evaluation, and shares what should be done to achieve
quality before starting with costly fine-tuning.
—Rafal Kawala, Senior AI Engineering Director, 16
years of experienc


#### A data inspection step to understand the overall size of the source text:

* full_text = " ".join(...): Iterates through the loaded pages and concatenates all the text into a single string.

**Purpose:** The output shows the raw document size is 1,080,309 characters. This confirms the content is too large for most LLM context windows, mandating the need for the chunking step that follows.

In [4]:
# Combine all text for exploration
full_text = " ".join([p.page_content for p in pages])
print(f"Document length: {len(full_text)} characters")
print(f"Sample snippet:\n{full_text[:600]}")

Document length: 1080309 characters
Sample snippet:
 Praise for AI Engineering
This book offers a comprehensive, well-structured guide to the
essential aspects of building generative AI systems. A must-read for
any professional looking to scale AI across the enterprise.
—Vittorio Cretella, former global CIO, P&G and Mars
Chip Huyen gets generative AI. On top of that, she is a remarkable
teacher and writer whose work has been instrumental in helping
teams bring AI into production. Drawing on her deep expertise, AI
Engineering serves as a comprehensive and holistic guide,
masterfully detailing everything required to design and deploy
generative A


## Split the document into Chunks

### This is the essential preparation for the retrieval step, optimizing the data for vector search:

1. splitter = RecursiveCharacterTextSplitter(...): Initializes the text splitting strategy.

2. chunk_size=1000: Sets the maximum size for each unit of text (chunk) at 1,000 characters.

3. chunk_overlap=200: Ensures the last 200 characters of one chunk are included in the start of the next chunk. This is a best practice to preserve context continuity and improve retrieval accuracy.

4. chunks = splitter.split_documents(pages): Executes the splitting, creating the final list of smaller, context-aware document chunks.

**Purpose:** The result, 1,640 chunks, transforms the single large document into a large set of semantically meaningful, searchable units.

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(pages)

print(f" Created {len(chunks)} chunks from {len(pages)} pages.")
print(chunks[0].page_content[:400])

 Created 1640 chunks from 991 pages.
Praise for AI Engineering
This book offers a comprehensive, well-structured guide to the
essential aspects of building generative AI systems. A must-read for
any professional looking to scale AI across the enterprise.
—Vittorio Cretella, former global CIO, P&G and Mars
Chip Huyen gets generative AI. On top of that, she is a remarkable
teacher and writer whose work has been instrumental in helping



In [6]:
print(f"Total chunks: {len(chunks)}")

Total chunks: 1640


### Access a specific chunk

#### These final cells serve only to confirm the data preparation steps succeeded:

* Accessing/Printing Chunks: Prints the first 400 characters of the first chunk, the total number of chunks, and the content of specific chunks by index (e.g., index 2, index 4, index 12).

**Purpose:** To verify that the chunks array is correctly populated and contains clean, readable, segmented text, ensuring the data is ready for the subsequent RAG steps (embedding and indexing).


In [7]:
# First chunk
print(chunks[0].page_content)

print("\n")

# Fifth chunk
print(chunks[4].page_content)

print("\n")

# 13th chunk
print(chunks[12].page_content)

Praise for AI Engineering
This book offers a comprehensive, well-structured guide to the
essential aspects of building generative AI systems. A must-read for
any professional looking to scale AI across the enterprise.
—Vittorio Cretella, former global CIO, P&G and Mars
Chip Huyen gets generative AI. On top of that, she is a remarkable
teacher and writer whose work has been instrumental in helping
teams bring AI into production. Drawing on her deep expertise, AI
Engineering serves as a comprehensive and holistic guide,
masterfully detailing everything required to design and deploy
generative AI applications in production.
—Luke Metz, cocreator of ChatGPT, former research
manager at OpenAI
Every AI engineer building real-world applications should read this
book. It’s a vital guide to end-to-end AI system design, from model
development and evaluation to large-scale deployment and operation.
—Andrei Lopatenko, Director Search and AI, Neuron7


AI Engineering
Building Applications with Foun

### Loop Through all chunks

If you want to quickly see the first 200 characters of each chunk:

In [43]:
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk.page_content[:200])
    print("\n")
    if i == 9:
        break

--- Chunk 1 ---
Praise for AI Engineering
This book offers a comprehensive, well-structured guide to the
essential aspects of building generative AI systems. A must-read for
any professional looking to scale AI acros


--- Chunk 2 ---
This book serves as an essential guide for building AI products that
can scale. Unlike other books that focus on tools or current trends
that are constantly changing, Chip delivers timeless foundation


--- Chunk 3 ---
AI Engineering is a practical guide that provides the most up-to-date
information on AI development, making it approachable for novice
and expert leaders alike. This book is an essential resource for



--- Chunk 4 ---
AI Engineering is a comprehensive guide that serves as an essential
reference for both understanding and implementing AI systems in
practice.
—Han Lee, Director—Data Science, Moody’s
AI Engineering is


--- Chunk 5 ---
AI Engineering
Building Applications with Foundation Models
Chip Huyen
OceanofPDF .com


--- Chunk 6 ---
AI 

### Access a chunk for later use

In [9]:
example_chunk = chunks[2].page_content
print(example_chunk)

AI Engineering is a practical guide that provides the most up-to-date
information on AI development, making it approachable for novice
and expert leaders alike. This book is an essential resource for
anyone looking to build robust and scalable AI systems.
—Vicki Reyzelman, Chief AI Solutions Architect,
Mave Sparks


## Create Embeddings

We’ll use a pre-trained sentence embedding model from Hugging Face — "sentence-transformers/all-MiniLM-L6-v2" (lightweight and fast).

In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Test embedding creation on one sample chunk
test_embedding = embedding_model.embed_query(chunks[0].page_content)
print(f" Sample embedding vector length: {len(test_embedding)}")

/tmp/ipython-input-599179408.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Sample embedding vector length: 384


#### We’ll encode each chunk into a vector representation.

In [11]:
texts = [c.page_content for c in chunks]
embeddings = embedding_model.embed_documents(texts)

print(" Generated embeddings with shape:", len(embeddings), "x", len(embeddings[0]))

 Generated embeddings with shape: 1640 x 384


#### embed_documents() returns a list of embeddings rather than a NumPy array, so before adding to FAISS you’ll need:



In [12]:
import numpy as np
embeddings = np.array(embeddings, dtype = "float32")

## Build a FAISS Vector Store

FAISS (Facebook AI Similarity Search) helps us store and quickly find similar vectors (chunks).

In [13]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # inner product = cosine similarity for normalized vectors
index.add(embeddings.astype("float32"))

print(f" FAISS index created with {index.ntotal} vectors.")

 FAISS index created with 1640 vectors.


## Save the FAISS Index & Metadata

We’ll store:

	•	The FAISS binary index file
	•	The chunk metadata (page numbers, chunk IDs, etc.)

In [14]:
import json

os.makedirs("faiss_index_ai_engineering", exist_ok=True)
faiss.write_index(index, "faiss_index_ai_engineering/index.faiss")

metadata = [c.metadata for c in chunks]
with open("faiss_index_ai_engineering/metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(" Saved FAISS index and metadata locally.")

 Saved FAISS index and metadata locally.


## Test Semantic Search

Let’s test if your RAG base is working properly.

In [15]:
def semantic_search(query, k=3):
    # Use LangChain method for embeddings
    query_emb = embedding_model.embed_query(query)
    query_emb = np.array([query_emb], dtype="float32")

    # Perform FAISS search
    distances, indices = index.search(query_emb, k)

    # Collect results
    results = []
    for i, idx in enumerate(indices[0]):
        text = chunks[idx].page_content[:400].replace("\n", " ")
        page = chunks[idx].metadata.get("page", "Unknown")
        results.append({"rank": i+1, "page": page, "text": text})

    return results

### Define your queries

These are natural language questions you want to ask your document (in this case, Chip Huyen’s AI Engineering book).

In [16]:
# Example user queries
queries = [
    "What is AI engineering?",
    "How does the book describe model deployment?",
    "What are some challenges in real-world machine learning systems?",
]

###Run the search for each query

We’ll loop through each question and show the top k (default 3) most relevant chunks.

In [17]:
for q in queries:
    print(f"\n Query: {q}\n" + "-"*80)
    results = semantic_search(q, k=3)

    for r in results:
        print(f" Rank {r['rank']} | Page {r['page']}")
        print(r['text'])
        print("-"*80)


 Query: What is AI engineering?
--------------------------------------------------------------------------------
 Rank 1 | Page 5
AI Engineering by Chip Huyen Copyright © 2025 Developer Experience Advisory LLC. All rights reserved. Printed in the United States of America. Published by O’Reilly Media, Inc., 1005 Gravenstein Highway North, Sebastopol, CA 95472. O’Reilly books may be purchased for educational, business, or sales promotional use. Online editions are also available for most titles (http://oreilly.com). For more i
--------------------------------------------------------------------------------
 Rank 2 | Page 2
AI Engineering is a practical guide that provides the most up-to-date information on AI development, making it approachable for novice and expert leaders alike. This book is an essential resource for anyone looking to build robust and scalable AI systems. —Vicki Reyzelman, Chief AI Solutions Architect, Mave Sparks
------------------------------------------------------

In [18]:
!pip install google-generativeai sentence-transformers

In [34]:
!pip show google-generativeai

Name: google-generativeai
Version: 0.8.5
Summary: Google Generative AI High level API client library and tools.
Home-page: https://github.com/google/generative-ai-python
Author: Google LLC
Author-email: googleapis-packages@google.com
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: google-ai-generativelanguage, google-api-core, google-api-python-client, google-auth, protobuf, pydantic, tqdm, typing-extensions
Required-by: 


## Configure Gemini API key

In [20]:

from sentence_transformers import SentenceTransformer
import google.generativeai as genai
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")
genai.configure(api_key=os.environ["GEMINI_API_KEY"])

Enter your Gemini API key: ··········


## Load FAISS index and chunk data (from Day 23)

In [22]:
FAISS_INDEX_DIR = "faiss_index_ai_engineering"

# Load FAISS index
index = faiss.read_index(os.path.join(FAISS_INDEX_DIR, "index.faiss"))

# Load metadata
with open(os.path.join(FAISS_INDEX_DIR, "metadata.json"), "r", encoding="utf-8") as f:
    metadatas = json.load(f)

texts = [chunk.page_content for chunk in chunks]

with open(os.path.join(FAISS_INDEX_DIR, "texts.txt"), "w", encoding="utf-8") as f:
    f.write("\n\n".join(texts))
# Load texts
with open(os.path.join(FAISS_INDEX_DIR, "texts.txt"), "r", encoding="utf-8") as f:
    all_texts = f.read().split("\n\n")

print(f"Loaded FAISS index and {len(all_texts)} text chunks.")

Loaded FAISS index and 1640 text chunks.


 ## Retrieve top relevant chunks for a user query

In [28]:
def retrieve_chunks(query, k=3):

    query_emb = embedding_model.embed_query(query)
    query_emb = np.array([query_emb], dtype="float32")

    # Search top-k similar chunks in FAISS index
    distances, indices = index.search(query_emb, k)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if 0 <= idx < len(all_texts):
            text = all_texts[idx].replace("\n", " ")
            meta = metadatas[idx]
            results.append({
                "score": float(dist),
                "text": text,
                "page": meta.get("page", "Unknown"),
                "chunk_id": meta.get("chunk_id", idx)
            })
    return results

	•	Converts the query into the same embedding format.
	•	Searches in FAISS for the k most similar chunks.
	•	Returns those chunks + metadata for reference.


## Build a RAG prompt

In [39]:
def build_prompt(query, retrieved_chunks):
    context = ""
    for r in retrieved_chunks:
        context += f"[Page {r['page']}, Chunk {r['chunk_id']}]\n{r['text']}\n\n"

    prompt = f"""
You are a helpful assistant. Use the provided context to answer the user's question. if you don't find it Look Again"

Context:
{context}

Question: {query}

Answer:
"""
    return prompt

## Generate the answer using Gemini

In [40]:
def generate_answer(prompt):
    model = genai.GenerativeModel("gemini-2.5-flash")
    response = model.generate_content(prompt)
    return response.text

## Create a complete RAG pipeline

In [41]:
def rag_query(query, k=3):
    retrieved = retrieve_chunks(query, k)
    prompt = build_prompt(query, retrieved)
    answer = generate_answer(prompt)

    print(f" Question: {query}\n")
    print(" Answer:", answer, "\n")
    print(" Sources:")
    for r in retrieved:
        print(f"- Page: {r['page']} | Chunk: {r['chunk_id']} | Score: {r['score']:.4f}")

## Test Queries for RAG

In [42]:
# List of 10 test queries
test_queries = [
    "What is AI engineering and how does it differ from traditional software engineering?",
    "How can machine learning models be deployed in production safely?",
    "What are the key components of a scalable ML data pipeline?",
    "How do you monitor ML models in production for performance drift?",
    "What are the best practices for feature engineering in AI projects?",
    "How should AI engineers structure experiments to validate models?",
    "What are the main ethical considerations in AI system design?",
    "Which tools and platforms are recommended for scalable AI infrastructure?",
    "What are common MLOps practices for maintaining ML workflows?",
    "Give examples of real-world applications of AI engineering principles."
]

# Loop through the queries
for i, query in enumerate(test_queries, 1):
    print(f"\n Test Query {i}: {query}\n")
    rag_query(query, k=3)  # Calls your existing RAG pipeline
    print("------------------------------------------------------------")


 Test Query 1: What is AI engineering and how does it differ from traditional software engineering?

 Question: What is AI engineering and how does it differ from traditional software engineering?

 Answer: The provided context discusses the relationship and differences between AI engineering and traditional ML engineering, noting that their roles have significant overlap and some companies put them under the same umbrella. It also states that a following section will further break down how AI engineering differs from traditional ML engineering.

However, the provided context does not explicitly define what AI engineering is, nor does it detail how AI engineering differs from traditional **software engineering**. 

 Sources:
- Page: 13 | Chunk: 17 | Score: 0.6753
- Page: 91 | Chunk: 146 | Score: 0.6641
- Page: 5 | Chunk: 5 | Score: 0.6557
------------------------------------------------------------

 Test Query 2: How can machine learning models be deployed in production safely?

 Que

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3401.01ms


 Question: How should AI engineers structure experiments to validate models?

 Answer: AI engineers should structure experiments to validate models through systematic experimentation, starting with prompting.

Here's a breakdown of the approach:
*   **Systematic Experimentation:** The process should be systematic, not minimal or unsystematic.
*   **Start with Prompting:** Begin experiments with prompting, following best practices. Only explore more advanced solutions if prompting alone proves inadequate.
*   **Thorough Prompt Testing:** Ensure various prompts are thoroughly tested, as model performance can vary greatly. Refine the prompt experiment process to ensure instructions are clear, examples represent actual data, and metrics are well-defined.
*   **Experiment with Various Elements:** Beyond prompts, experiment with different models, retrieval algorithms, and sampling variables.
*   **Establish an Evaluation Pipeline:** Set up a solid evaluation pipeline to help detect failures 

## Analysis of the Notebook and Results

This notebook successfully implements a basic Retrieval-Augmented Generation (RAG) pipeline using a PDF document. Here's a breakdown of the key steps and observations:

**1. Setup and Dependencies:**

*   The notebook starts by installing necessary libraries (`langchain`, `sentence-transformers`, `faiss-cpu`, `pypdf`, etc.). This is a standard and correct approach for setting up the environment.
*   Importing the required modules is also done correctly.

**2. Data Loading and Preprocessing:**

*   The `PyPDFLoader` is used to load the PDF document. The output shows that 991 pages were loaded, indicating successful file reading.
*   Combining the text from all pages into a single string and checking its length confirms that the document is indeed large and requires splitting.
*   The `RecursiveCharacterTextSplitter` is used to break down the document into smaller chunks with a specified `chunk_size` and `chunk_overlap`. This is a crucial step for managing the context length of the language model and ensuring semantic continuity between chunks. The creation of 1640 chunks seems reasonable for a document of this size.
*   Inspecting individual chunks confirms that the splitting process worked as expected.

**3. Embedding Creation:**

*   The notebook uses `HuggingFaceEmbeddings` with the "sentence-transformers/all-MiniLM-L6-v2" model to create embeddings for each chunk. This is a good choice for its balance of performance and efficiency.
*   The warning about `LangChainDeprecationWarning` for `HuggingFaceEmbeddings` is noted. While the current code still works, it's good practice to update to the recommended `langchain_huggingface` package in the future.
*   Creating and checking the shape of the embeddings confirms that vectors of the expected dimension (384) were generated for all 1640 chunks.
*   Converting the embeddings to `float32` NumPy array is necessary for compatibility with FAISS.

**4. FAISS Vector Store:**

*   A FAISS `IndexFlatIP` is created, which is suitable for cosine similarity search when vectors are normalized (as is the case with Sentence Transformers).
*   Adding the embeddings to the index is done correctly.
*   The confirmation that the FAISS index contains 1640 vectors verifies that all chunks were indexed.

**5. Saving and Loading the Index and Metadata:**

*   Saving the FAISS index and metadata (`metadata.json` and `texts.txt`) is a good practice for persistence, allowing the index to be reused without re-processing the entire document.
*   Loading the saved index, metadata, and texts confirms that the persistence mechanism works.

**6. Semantic Search and RAG Pipeline:**

*   The `semantic_search` function correctly uses the embedding model and the FAISS index to retrieve the top-k most similar chunks based on a query.
*   The `build_prompt` function constructs a prompt for the language model, including the original query and the retrieved context. This is the core of the RAG approach.
*   The `generate_answer` function uses the Gemini 2.5 Flash model to generate a response based on the provided prompt.
*   The `rag_query` function orchestrates the retrieval and generation steps.
*   The initial test query "What are the key principles of AI engineering?" returned "I don’t know from the provided document." while showing relevant chunks in the sources. This suggests that the retrieved chunks might not contain the explicit answer in a way the model can easily extract, or the model's ability to synthesize the answer from the given chunks is limited.
*   The subsequent test queries show mixed results. Some queries (e.g., "How can machine learning models be deployed in production safely?", "How do you monitor ML models in production for performance drift?", "How should AI engineers structure experiments to validate models?", "What are the main ethical considerations in AI system design?") receive relevant answers extracted from the document, indicating that for certain topics, the RAG pipeline works well.
*   Other queries (e.g., "What is AI engineering and how does it differ from traditional software engineering?", "What are the key components of a scalable ML data pipeline?", "What are the best practices for feature engineering in AI projects?", "Which tools and platforms are recommended for scalable AI infrastructure?", "What are common MLOps practices for maintaining ML workflows?", "Give examples of real-world applications of AI engineering principles.") still result in "I don’t know from the provided document." This suggests that either the retrieval is not bringing back the most relevant chunks for these specific questions, or the model is unable to synthesize the answer from the retrieved information.

## Conclusion and Summary

The notebook provides a solid foundation for a RAG pipeline. It successfully loads, preprocesses, embeds, and indexes a large PDF document. The basic semantic search and RAG query functions are implemented correctly.

However, the performance of the RAG pipeline in answering some of the test queries is limited, with several questions resulting in "I don’t know from the provided document." This indicates areas for improvement in either the retrieval or the generation stage of the pipeline.

**Summary:**

*   Basic RAG pipeline successfully implemented.
*   PDF loaded, chunked, embedded, and indexed in FAISS.
*   Semantic search retrieves relevant chunks.
*   Gemini 2.5 Flash model used for generation.
*   Performance on some queries is not optimal, highlighting the need for further refinement.



## Day 25 Plan: Improving the RAG Pipeline

On Day 25, we will focus on improving the performance of the RAG pipeline by exploring and implementing techniques to enhance both the retrieval and the generation stages. Here is a plan:

1.  **Address Deprecation Warning:** Update the `HuggingFaceEmbeddings` import to use the recommended `langchain_huggingface` package.
2.  **Experiment with Different Embedding Models:** The choice of embedding model significantly impacts retrieval accuracy. We will explore using a potentially more powerful or domain-specific embedding model to see if it improves the relevance of retrieved chunks for the challenging queries.
3.  **Optimize Chunking Strategy:** The current chunk size and overlap are fixed. We can experiment with different `chunk_size` and `chunk_overlap` values to see if a different chunking strategy leads to better retrieval. We could also explore more advanced text splitting techniques.
4.  **Implement Re-ranking:** Even if the initial retrieval brings back relevant chunks, the most relevant information might not be in the top-k results. Implementing a re-ranking step using a dedicated re-ranking model can help prioritize the truly most relevant chunks for the language model.
5.  **Refine Prompt Engineering:** While the current prompt is basic, we can experiment with different prompt templates to guide the language model more effectively in synthesizing answers from the retrieved context. This could involve techniques like including instructions on how to use the provided context and what to do if the answer is not found.
6.  **Explore Different Language Models:** The Gemini 2.5 Flash model is efficient but might not be as capable as larger models in synthesizing complex answers. We could experiment with a more powerful Gemini model (if available and suitable for the task) to see if it improves the generation quality.
7.  **Evaluate with More Diverse Queries:** Expand the set of test queries to cover a wider range of topics and question types from the document to get a more comprehensive understanding of the pipeline's performance.
8.  **Analyze Retrieval Results for Failing Queries:** For the queries that resulted in "I don’t know from the provided document," we will manually inspect the retrieved chunks to understand why the pipeline failed. This will help identify whether the issue is in retrieval or generation.
9.  **Add Error Handling and Logging:** Implement more robust error handling and logging to better understand issues during the pipeline execution.
10. **Finish task**: Summarize the improvements made and their impact on the RAG pipeline's performance.

This plan will allow us to systematically improve the RAG pipeline and address the current limitations observed in the test queries.